<a href="https://colab.research.google.com/github/paulinepiccio/cinema-and-war/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Imports
import requests
import pandas as pd
import time
from google.colab import userdata

# Configuration
API_KEY = userdata.get('TMDB_API_KEY')
BASE_URL = "https://api.themoviedb.org/3"
WAR_GENRE_ID = 10752  # TMDB code for the "War" genre

# Our 6 countries ISO codes
COUNTRIES = {
    'US': 'United States',
    'FR': 'France',
    'GB': 'United Kingdom',
    'DE': 'Germany',
    'RU': 'Russia',
    'SU': 'Soviet Union', # pre-1991 soviet productions
    'JP': 'Japan'
}

START_DATE = "1939-01-01"
END_DATE = "2025-12-31"

print(f"Configuration OK : {'Yes' if API_KEY else 'No'}")
print(f"{len(COUNTRIES)} countries to analyse")

Configuration OK : Yes
7 countries to analyse


In [2]:
# Test
test_url = f"{BASE_URL}/discover/movie"
test_params = {
    'api_key': API_KEY,
    'with_genres': WAR_GENRE_ID,
    'with_origin_country': 'US',
    'primary_release_date.gte': START_DATE,
    'primary_release_date.lte': END_DATE,
    'language': 'en-US',
    'page': 1
}

response = requests.get(test_url, params=test_params)
data = response.json()

print(f"Status code : {response.status_code}")
print(f"Total movies found (US) : {data.get('total_results', 0)}")
print(f"Total pages : {data.get('total_pages', 0)}")
print(f"\nMost popular movie :")
if data.get('results'):
    first = data['results'][0]
    print(f"  Title : {first['title']}")
    print(f"  Date  : {first['release_date']}")
    print(f"  Note  : {first['vote_average']}")

Status code : 200
Total movies found (US) : 3506
Total pages : 176

Most popular movie :
  Title : Schindler's List
  Date  : 1993-12-15
  Note  : 8.568


In [3]:
def fetch_war_movies(country_code, country_name):

    all_movies = []
    page = 1

    while True:
        params = {
            'api_key': API_KEY,
            'with_genres': WAR_GENRE_ID,
            'with_origin_country': country_code,
            'primary_release_date.gte': START_DATE,
            'primary_release_date.lte': END_DATE,
            'language': 'en-US',
            'page': page
        }

        response = requests.get(f"{BASE_URL}/discover/movie", params=params)

        if response.status_code != 200:
            print(f"Error page {page} : {response.status_code}")
            break

        data = response.json()
        results = data.get('results', [])

        if not results:
            break

        for movie in results:
            movie['origin_country_code'] = country_code
            movie['origin_country_name'] = country_name

        all_movies.extend(results)

        total_pages = data.get('total_pages', 1)
        print(f"Page {page}/{total_pages} — {len(results)} films fetched")

        if page >= total_pages:
            break

        page += 1
        time.sleep(0.25)  # rate limit TMDB

    return all_movies

print("Function fetch_war_movies defined")

Function fetch_war_movies defined


In [4]:
all_movies_data = []

for country_code, country_name in COUNTRIES.items():
    print(f"\nCollect for {country_name} ({country_code})...")
    movies = fetch_war_movies(country_code, country_name)
    all_movies_data.extend(movies)
    print(f"Total {country_name} : {len(movies)} films")

print(f"\n{'='*50}")
print(f"GRAND TOTAL : {len(all_movies_data)} films fetched")
print(f"{'='*50}")


Collect for United States (US)...
Page 1/176 — 20 films fetched
Page 2/176 — 20 films fetched
Page 3/176 — 20 films fetched
Page 4/176 — 20 films fetched
Page 5/176 — 20 films fetched
Page 6/176 — 20 films fetched
Page 7/176 — 20 films fetched
Page 8/176 — 20 films fetched
Page 9/176 — 20 films fetched
Page 10/176 — 20 films fetched
Page 11/176 — 20 films fetched
Page 12/176 — 20 films fetched
Page 13/176 — 20 films fetched
Page 14/176 — 20 films fetched
Page 15/176 — 20 films fetched
Page 16/176 — 20 films fetched
Page 17/176 — 20 films fetched
Page 18/176 — 20 films fetched
Page 19/176 — 20 films fetched
Page 20/176 — 20 films fetched
Page 21/176 — 20 films fetched
Page 22/176 — 20 films fetched
Page 23/176 — 20 films fetched
Page 24/176 — 20 films fetched
Page 25/176 — 20 films fetched
Page 26/176 — 20 films fetched
Page 27/176 — 20 films fetched
Page 28/176 — 20 films fetched
Page 29/176 — 20 films fetched
Page 30/176 — 20 films fetched
Page 31/176 — 20 films fetched
Page 32/176 —

In [5]:
# Conversion into DataFrame
df = pd.DataFrame(all_movies_data)
print(f"Dimensions : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns available :")
print(df.columns.tolist())
print(f"\n5 first rows :")
df.head()

Dimensions : 7123 rows × 17 columns

Columns available :
['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count', 'origin_country_code', 'origin_country_name']

5 first rows :


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count,origin_country_code,origin_country_name
0,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,25.9785,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17407,US,United States
1,False,/d3z8MH9OvTOOSxy5QwAG0cS6GKU.jpg,"[10752, 28, 36]",652,Troy,en,Troy,"In year 1250 B.C. during the late Bronze age, ...",20.4345,/51auXjXepW1zblzhaN7CAhwvf5i.jpg,2004-05-13,False,False,7.170,11002,US,United States
2,False,/vDKRMZGFTKP9nQolzeSB1rB1w6p.jpg,"[18, 36, 10752]",324786,Hacksaw Ridge,en,Hacksaw Ridge,"WWII American Army Medic Desmond T. Doss, who ...",16.8056,/fnOMP6mjmOmZwmlC1n0K7ivrzt1.jpg,2016-10-07,False,False,8.190,14803,US,United States
3,False,/kP8rK9dGS1pr0HrnmXfIi2heWjo.jpg,"[18, 28, 12, 36, 10752]",1495,Kingdom of Heaven,en,Kingdom of Heaven,"After his wife dies, a blacksmith named Balian...",20.2622,/rNaBe4TwbMef71sgscqabpGKsxh.jpg,2005-05-03,False,False,7.030,4915,US,United States
4,False,/hwNtEmmugU5Yd7hpfprNWI0DGIn.jpg,"[18, 53, 10752]",16869,Inglourious Basterds,en,Inglourious Basterds,"In Nazi-occupied France during World War II, a...",17.6879,/7sfbEnaARXDDhKm0CZ7D7uc2sbo.jpg,2009-08-02,False,False,8.217,24106,US,United States


In [6]:
# Movies per country
print("Breakdown by country of origin")
print(df['origin_country_name'].value_counts())

# Movies per decade
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['decade'] = (df['release_year'] // 10 * 10).astype('Int64')

print("\nBreakdown by decade")
print(df['decade'].value_counts().sort_index())

Breakdown by country of origin
origin_country_name
United States     3506
United Kingdom     878
France             823
Soviet Union       812
Japan              456
Germany            362
Russia             286
Name: count, dtype: int64

Breakdown by decade
decade
1930      53
1940     995
1950     539
1960     655
1970     506
1980     608
1990     424
2000     895
2010    1451
2020     997
Name: count, dtype: Int64


In [7]:
columns_to_keep = [
    'id', 'title', 'original_title', 'original_language',
    'release_date', 'release_year', 'decade',
    'overview', 'popularity', 'vote_average', 'vote_count',
    'origin_country_code', 'origin_country_name', 'genre_ids'
]

df_clean = df[columns_to_keep].copy()

output_path = 'war_movies_raw.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"Downloaded: {output_path}")
print(f"Size : {df_clean.shape[0]} movies, {df_clean.shape[1]} columns")

Downloaded: war_movies_raw.csv
Size : 7123 movies, 14 columns


In [8]:
from google.colab import files
files.download('war_movies_raw.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>